In [1]:
import random
from datetime import datetime
from datetime import timedelta

import duckdb
import pandas as pd
from faker import Faker

In [2]:
Faker.seed(42)
fake = Faker(['ko_KR', 'en_US'])


def generate_fake_data(n: int):
    return [{
        "id": i + 1,
        "name": fake.name(),
        "age": fake.random_int(min=20, max=65),
        "weight": fake.random_int(min=10, max=250),
        "height": fake.random_int(min=10, max=250),
        "gender": random.choice(["남성", "여성"]),
        "address": fake.address(),
        "job": fake.job(),
        "email": fake.email(),
        "signup": (datetime.now() - timedelta(days=random.randint(0, 365))).date().isoformat()
    } for i in range(n)]

In [3]:
connect = duckdb.connect()
connect.execute("INSTALL httpfs;")
connect.execute("LOAD httpfs;")
connect.execute("SET s3_endpoint='minio:9000';")
connect.execute("SET s3_use_ssl=false;")
connect.execute("SET s3_url_style='path'")

### Pandas ↔ DuckDB

In [4]:
df = pd.DataFrame({"id": [1, 2, 3], "name": ["A", "B", "C"], "value": [10, 20, 30]})
connect.register("df_tbl", df)
result = connect.execute("SELECT name, value * 2 AS value2 FROM df_tbl WHERE value >= 20").df()
print(result)

  name  value2
0    B      40
1    C      60


### SQL 결과 → Pandas

In [5]:
out_df = connect.execute("SELECT sum(value) AS total FROM df_tbl").df()
print(out_df)

   total
0   60.0


### S3 파일 쓰기

In [6]:
for i in range(10):
    dataframe = pd.DataFrame(generate_fake_data(100))
    connect.register("dataframe", dataframe)
    s3_url = f"s3://mmix-prod-dataengineer-datalakehouse/sample/customers/customer_{i}.parquet"
    connect.execute(f"COPY dataframe TO '{s3_url}' (FORMAT PARQUET, COMPRESSION ZSTD);")

### S3 파일 읽기

In [7]:
s3_url = "s3://mmix-prod-dataengineer-datalakehouse/sample/customers/*.parquet"
cursor = connect.execute(f"""SELECT id, name, age, weight, height, gender, address, job, email, signup FROM read_parquet('{s3_url}') LIMIT 1""")
cols = [c[0] for c in cursor.description]
rows = cursor.fetchall()

### DuckDB 맛보기

In [8]:
connect.register("customers", pd.DataFrame(generate_fake_data(10000)))

In [9]:
# 2) 성별 통계(인원/평균나이/키/몸무게)
connect.execute("SELECT gender, COUNT(*) AS cnt, AVG(age) AS avg_age, AVG(height) AS avg_height, AVG(weight) AS avg_weight FROM customers GROUP BY gender ORDER BY cnt DESC").fetchdf()

,gender,cnt,avg_age,avg_height,avg_weight
0,여성,5058,42.670819,130.787663,128.889482
1,남성,4942,42.431202,130.229259,129.689802


In [10]:
# 3) 직업 Top 10(인원)
connect.execute("SELECT job, COUNT(*) AS cnt FROM customers GROUP BY job ORDER BY cnt DESC LIMIT 10").fetchdf()

,job,cnt
0,병무행정 사무원,22
1,금속 / 재료공학 연구원 및 기술자,21
2,성직자,21
3,제품 및 광고 영업원,20
4,승강기 설치 및 정비원,20
5,중식 주방장 및 조리사,20
6,내선전공,20
7,무역 사무원,19
8,정부 및 공공 행정 전문가,19
9,음향 및 녹음 기사,19


In [11]:
# 4) 가입 월별 추이(YYYY-MM)
connect.execute("SELECT strftime(CAST(signup AS DATE), '%Y-%m') AS ym, COUNT(*) AS cnt FROM customers GROUP BY ym ORDER BY ym").fetchdf()

,ym,cnt
0,2025-01,371
1,2025-02,726
2,2025-03,854
3,2025-04,810
4,2025-05,825
5,2025-06,893
6,2025-07,840
7,2025-08,843
8,2025-09,799
9,2025-10,876


In [12]:
# 5) 나이대(age bucket) + 성별 분포
connect.execute(
    """
    SELECT gender,
           CASE
               WHEN age BETWEEN 20 AND 29 THEN '20s'
               WHEN age BETWEEN 30 AND 39 THEN '30s'
               WHEN age BETWEEN 40 AND 49 THEN '40s'
               WHEN age BETWEEN 50 AND 59 THEN '50s'
               ELSE '60s' END AS age_group,
           COUNT(*)           AS cnt
    FROM customers
    GROUP BY gender, age_group
    ORDER BY gender, age_group
    """).fetchdf()

,gender,age_group,cnt
0,남성,20s,1086
1,남성,30s,1083
2,남성,40s,1035
3,남성,50s,1120
4,남성,60s,618
5,여성,20s,1046
6,여성,30s,1122
7,여성,40s,1125
8,여성,50s,1121
9,여성,60s,644


In [13]:
# 6) BMI 계산 후, 성별 평균 BMI
connect.execute("SELECT gender, AVG(weight / POW(height/100.0, 2)) AS avg_bmi FROM customers GROUP BY gender").fetchdf();